In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rioxarray as rxr
from pathlib import Path
import sys

# path to your repo root (where `src` lives)
project_root = Path("/path/to/project")

# add it to sys.path if not already there
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.sea_ice_drift.adapted_dualPol import warp_image_with_flow, warp_with_forward_flow, warp_image_with_flow_anchor

In [ ]:
def plot_drift(npz_file, quiver_step=80):
    """
    Visualize SAR drift from one NPZ vector field.
    
    Panels:
       [1] Start image (dB)
       [2] End image (dB)
       [3] Start image + vectors
       [4] Warped(start) using drift
    """
    
    npz_file = Path(npz_file)

    # Load drift + metadata
    vf = np.load(npz_file, allow_pickle=True)
    meta = vf["meta"].item()

    u = vf["u"]
    v = vf["v"]
    # print(u[500, 500], v[500, 500])

    start_tiff = Path(meta["start_path"])
    end_tiff   = Path(meta["end_path"])

    print(f"Plotting drift:")
    print(f"  Start image:  {start_tiff}")
    print(f"  End image:    {end_tiff}")

    # Load SAR images (HV band index 1)
    da1 = rxr.open_rasterio(start_tiff)
    da2 = rxr.open_rasterio(end_tiff)

    img1 = da1[0].values.astype(float)
    img2 = da2[0].values.astype(float)

    # Warp image1 with the drift field
    # img1_warp = warp_image_with_flow(img1, u, v)
    img1_warp = warp_with_forward_flow(img1, u, v)
    # img1_warp = warp_image_with_flow_anchor(img1, u, v, anchor_dx=50, anchor_dy=-50)


    # Prepare quiver grid
    H, W = u.shape
    yy = np.arange(0, H, quiver_step)
    xx = np.arange(0, W, quiver_step)
    Xq, Yq = np.meshgrid(xx, yy)
    Uq = u[yy[:,None], xx]
    Vq = v[yy[:,None], xx]

    # ---- Plot figures ---- #
    fig, axes = plt.subplots(2, 2, figsize=(22, 14))
    ax_start, ax_end, ax_vec, ax_warp = axes.flatten()

    # 1 — Start image in dB
    ax_start.imshow(10*np.log10(img1), cmap="gray")
    ax_start.set_title(f"Start image (t0): {start_tiff.stem}")

    
    # 2 — End image in dB
    ax_end.imshow(10*np.log10(img2), cmap="gray")
    ax_end.set_title(f"End image (t1): {end_tiff.stem}")

    # 3 — Vector field on start image
    ax_vec.imshow(10*np.log10(img1), cmap="gray")
    ax_vec.quiver(
        Xq, Yq, Uq, Vq,
        color="red",
        scale_units="xy",
        scale=1, # need to change to 0.001 or so if plotting velocities
        angles="xy",
        width=0.003
    )
    ax_vec.set_title("Start image + vector field")

    # 4 — Warped start image
    ax_warp.imshow(10*np.log10(img1_warp), cmap="gray")
    ax_warp.set_title("Warped start image using drift field")

    # Remove axis ticks
    for ax in axes.flatten():
        # ax.set_xticks([])
        # ax.set_yticks([])
        ax.grid(color="red", alpha=1, linestyle="--", linewidth=0.5)

    plt.tight_layout()
    plt.show()


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import rioxarray as rxr

def plot_drift_divergence(npz_file, pixel_size_m=100.0, cmap="RdBu_r", clip_percentile=99):
    """
    Visualize SAR drift divergence from one NPZ vector field.

    Panels:
       [1] Start image (dB)
       [2] End image (dB)
       [3] Start image + divergence (1/s)
       [4] Warped(start) using drift

    Notes:
      - If u,v are m/s, divergence returned is 1/s when pixel_size_m is in meters.
      - If your u,v are pixels (displacements), divergence will not be physical unless you convert first.
    """
    npz_file = Path(npz_file)

    # Load drift + metadata
    vf = np.load(npz_file, allow_pickle=True)
    meta = vf["meta"].item() if "meta" in vf else {}

    u = vf["u"].astype(np.float32, copy=False)
    v = vf["v"].astype(np.float32, copy=False)

    # Prefer pixel size from meta if available
    px_m = float(meta.get("pixel_size_m", pixel_size_m))

    start_tiff = Path(meta["start_path"])
    end_tiff   = Path(meta["end_path"])

    print(f"Plotting divergence:")
    print(f"  Start image:  {start_tiff}")
    print(f"  End image:    {end_tiff}")
    print(f"  pixel_size_m: {px_m}")

    # Load SAR images
    da1 = rxr.open_rasterio(start_tiff)
    da2 = rxr.open_rasterio(end_tiff)
    img1 = da1[1].values.astype(float)
    img2 = da2[1].values.astype(float)

    # Warp image1 with the drift field (same as your existing function)
    # img1_warp = warp_image_with_flow(img1, u, v)
    img1_warp = warp_with_forward_flow(img1, u, v)

    # ---- Divergence (1/s) ----
    # np.gradient returns d/d(row), d/d(col) if you pass a 2D array.
    # So: du/dx = du/dcol / dx_m, dv/dy = dv/drow / dy_m
    # du_drow, du_dcol = np.gradient(u)
    # dv_drow, dv_dcol = np.gradient(v)

    # div = (du_dcol + dv_drow) / px_m  # 1/s if u,v are m/s and px_m in m
######################
    from scipy.ndimage import gaussian_filter

    sigma_px = 10  # try 1–3 pixels

    u_s = gaussian_filter(u, sigma=sigma_px, mode="nearest")
    v_s = gaussian_filter(v, sigma=sigma_px, mode="nearest")

    du_drow, du_dcol = np.gradient(u_s)
    dv_drow, dv_dcol = np.gradient(v_s)

    div = (du_dcol + dv_drow) / px_m
###################

    # Robust color limits
    finite = np.isfinite(div)
    if np.any(finite):
        vmax = np.nanpercentile(np.abs(div[finite]), clip_percentile)
        if vmax == 0 or not np.isfinite(vmax):
            vmax = np.nanmax(np.abs(div[finite])) if np.nanmax(np.abs(div[finite])) > 0 else 1.0
    else:
        vmax = 1.0

    vmin = -vmax

    # ---- Plot figures ---- #
    fig, axes = plt.subplots(2, 2, figsize=(22, 14))
    ax_start, ax_end, ax_div, ax_warp = axes.flatten()

    ax_start.imshow(10*np.log10(img1), cmap="gray")
    ax_start.set_title(f"Start image (t0): {start_tiff.stem}")

    ax_end.imshow(10*np.log10(img2), cmap="gray")
    ax_end.set_title(f"End image (t1): {end_tiff.stem}")

    # Divergence overlay
    # im = ax_div.imshow(div, cmap=cmap, vmin=vmin, vmax=vmax, alpha=1)
    # ax_div.imshow(10*np.log10(img1), cmap="gray", alpha=0.3)

    # ax_div.set_title("Divergence overlay")
    # cbar = fig.colorbar(im, ax=ax_div, fraction=0.046, pad=0.04)
    # cbar.set_label("1/s")
    # Normalize divergence magnitude to [0, 1]
    abs_div = np.abs(div)

    # Use same vmax as colormap
    alpha = abs_div / vmax
    alpha = np.clip(alpha, 0, 1)

    # Optional: nonlinear boost to emphasize strong features
    alpha = alpha**0.8   # try 0.5–1.0

    # Plot
    ax_div.imshow(10*np.log10(img1), cmap="gray", alpha=1.0)
    im = ax_div.imshow(div, cmap=cmap, vmin=vmin, vmax=vmax, alpha=alpha)

    ax_div.set_title("Divergence overlay")
    cbar = fig.colorbar(im, ax=ax_div, fraction=0.046, pad=0.04)
    cbar.set_label("1/s")


    ax_warp.imshow(10*np.log10(img1_warp), cmap="gray")
    ax_warp.set_title("Warped start image using drift field")

    for ax in axes.flatten():
        ax.grid(color="red", alpha=1, linestyle="--", linewidth=0.5)

    plt.tight_layout()
    plt.show()



In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import rioxarray as rxr
from scipy.ndimage import gaussian_filter

# def warp_with_forward_flow(img, u, v):
#     # """
#     # Minimal forward-warp placeholder (your original function likely different).
#     # For visualization only: do a simple nearest-neighbor forward mapping with clipping.
#     # """
#     # H, W = img.shape
#     # coords = np.indices((H, W)).astype(np.float32)
#     # r = coords[0]  # rows
#     # c = coords[1]  # cols

#     # # forward map: (r, c) -> (r + v, c + u) if u is in pixels horizontally, v vertically
#     # # Here assume u is x (cols), v is y (rows)
#     # r2 = (r + v).round().astype(int)
#     # c2 = (c + u).round().astype(int)

#     # out = np.full_like(img, np.nan)
#     # valid = (r2 >= 0) & (r2 < H) & (c2 >= 0) & (c2 < W)
#     # out[r2[valid], c2[valid]] = img[r[valid], c[valid]]
#     # # Fill remaining with original using simple nearest-neighbor where possible
#     # mask_nan = ~np.isfinite(out)
#     # out[mask_nan] = img[mask_nan]
#     # return out

def plot_drift_divergence_with_mask(npz_file, pixel_size_m=100.0, cmap="RdBu_r", clip_percentile=99, mask_q=0.90):
    """
    Visualize SAR drift divergence and show top-(1-mask_q) mask (e.g. mask_q=0.90 => top 10% kept).
    Panels:
       [1] Start image (dB)
       [2] End image (dB)
       [3] Divergence overlay (masked: shown only where |div| in top fraction)
       [4] Mask (binary) and warped image
    """
    npz_file = Path(npz_file)

    vf = np.load(npz_file, allow_pickle=True)
    meta = vf["meta"].item() if "meta" in vf else {}

    u = vf["u"].astype(np.float32, copy=False)
    v = vf["v"].astype(np.float32, copy=False)

    px_m = float(meta.get("pixel_size_m", pixel_size_m))

    start_tiff = Path(meta["start_path"])
    end_tiff   = Path(meta["end_path"])

    print(f"Plotting divergence and top {(1.0-mask_q)*100:.1f}% mask:")
    print(f"  Start image:  {start_tiff}")
    print(f"  End image:    {end_tiff}")
    print(f"  pixel_size_m: {px_m}")

    # Read SAR images (pick band 1 like original)
    da1 = rxr.open_rasterio(start_tiff)
    da2 = rxr.open_rasterio(end_tiff)
    img1 = da1[1].values.astype(float)
    img2 = da2[1].values.astype(float)

    # Warp image1 with the drift field for quick visual check
    img1_warp = warp_with_forward_flow(img1, u, v)

    # --- smooth then compute divergence (1/s) ---
    sigma_px = 10  # same as your snippet; reduce to 1-3 for less smoothing
    u_s = gaussian_filter(u, sigma=sigma_px, mode="nearest")
    v_s = gaussian_filter(v, sigma=sigma_px, mode="nearest")

    du_drow, du_dcol = np.gradient(u_s)
    dv_drow, dv_dcol = np.gradient(v_s)

    div = (du_dcol + dv_drow) / px_m    # 1/s if u,v in m/s and px_m in m

    # robust color limits for divergence
    finite = np.isfinite(div)
    if np.any(finite):
        vmax = np.nanpercentile(np.abs(div[finite]), clip_percentile)
        if vmax == 0 or not np.isfinite(vmax):
            vmax = np.nanmax(np.abs(div[finite])) if np.nanmax(np.abs(div[finite])) > 0 else 1.0
    else:
        vmax = 1.0
    vmin = -vmax

    # --- compute top-(1-q) mask from |div| per-sample ---
    abs_div = np.abs(div)
    thr = np.quantile(abs_div[finite], mask_q) if np.any(finite) else 0.0
    mask = np.zeros_like(abs_div, dtype=bool)
    mask[finite] = abs_div[finite] >= thr

    # show actual fraction kept
    frac = mask.mean()
    print(f"  Mask threshold (quantile {mask_q:.2f}) = {thr:.3e}  -> kept fraction = {frac*100:.2f}%")

    # alpha map: lighten weak regions; but we will show divergence only where mask True
    # create masked divergence for plotting: set elsewhere to 0 and alpha 0
    div_masked = np.full_like(div, np.nan)
    div_masked[mask] = div[mask]

    # Create figure with 2x2 layout
    fig, axes = plt.subplots(2, 2, figsize=(22, 14))
    ax_start, ax_div_overlay, ax_mask, ax_warp = axes.flatten()

    # Panel 1: start image
    ax_start.imshow(10*np.log10(img1), cmap="gray")
    ax_start.set_title(f"Start image (t0): {start_tiff.stem}")

    # Panel 2: divergence overlay but only where mask True (use alpha=1 on mask, 0 elsewhere)
    ax_div_overlay.imshow(10*np.log10(img1), cmap="gray", alpha=1.0)
    # show diverence colormap only where masked (use np.ma to mask NaNs)
    import numpy.ma as ma
    div_ma = ma.masked_invalid(div_masked)
    im2 = ax_div_overlay.imshow(div_ma, cmap=cmap, vmin=vmin, vmax=vmax, alpha=0.9, interpolation="nearest")
    ax_div_overlay.set_title(f"Divergence overlay (masked top {(1.0-mask_q)*100:.0f}%)")
    cbar2 = fig.colorbar(im2, ax=ax_div_overlay, fraction=0.046, pad=0.02)
    cbar2.set_label("1/s")

    # Panel 3: binary mask (red overlay) on top of start image to see mask shape
    ax_mask.imshow(10*np.log10(img1), cmap="gray")
    # show mask as transparent red overlay
    cmap_mask = plt.cm.Reds
    # create RGBA where mask True -> red, else transparent
    rgba = np.zeros((*mask.shape, 4), dtype=float)
    rgba[..., 0] = 1.0  # red channel
    rgba[..., 3] = 0.6 * mask.astype(float)  # alpha where mask True
    ax_mask.imshow(rgba, origin="upper", interpolation="nearest")
    ax_mask.set_title(f"Top {(1.0-mask_q)*100:.0f}% mask (red overlay)\nkept fraction={frac*100:.2f}%")

    # Panel 4: warped start image
    ax_warp.imshow(10*np.log10(img1_warp), cmap="gray")
    ax_warp.set_title("Warped start image using drift field")

    for ax in axes.flatten():
        ax.set_xticks([])
        ax.set_yticks([])

    plt.tight_layout()
    plt.show()

# Example usage:
# plot_drift_divergence_with_mask("/path/to/your/npz_file.npz", pixel_size_m=100.0, mask_q=0.90)

In [ ]:
# pixel displacement
past_file = '/path/to/SAR_sea_ice_dataset/VECTOR_FIELDS_24h_pairs/HV_HH/region-27_0-82_951-35_52-83_8/2014/11/07/20141106T0905__20141107T0807_past.npz'
future_file = '/path/to/SAR_sea_ice_dataset/VECTOR_FIELDS_24h_pairs/HV_HH/region-27_0-82_951-35_52-83_8/2014/11/07/20141107T0807__20141108T0848_future.npz'

In [ ]:
# just checking that bakcward is actually backwards
# past_file = "/path/to/SAR_sea_ice_dataset/VECTOR_FIELDS_24h_pairs_wBackwardPastDrift/HV_HH/region-27_0-82_951-35_52-83_8/2014/11/08/20141107T0807__20141108T0848_backward_past.npz"

In [ ]:
# past_file = '/path/to/SAR_sea_ice_dataset/VECTOR_FIELDS_24h_pairs/HV/region-33_07-82_34-40_1-83_297/2020/10/18/20201017T0637__20201018T0719_past.npz'
# future_file = '/path/to/SAR_sea_ice_dataset/VECTOR_FIELDS_24h_pairs/HV/region-33_07-82_34-40_1-83_297/2020/10/18/20201018T0719__20201019T0621_future.npz'

In [ ]:
# past_file = "/path/to/SAR_sea_ice_dataset/VECTOR_FIELDS_24h_pairs/HV/region-27_0-82_951-35_52-83_8/2020/05/18/20200517T0840__20200518T0743_past.npz"
# future_file = "/path/to/SAR_sea_ice_dataset/VECTOR_FIELDS_24h_pairs/HV/region-27_0-82_951-35_52-83_8/2020/05/18/20200518T0743__20200519T0645_future.npz"

In [ ]:
# # velocity m/s
# past_file = '/path/to/SAR_sea_ice_dataset/VECTOR_FIELDS_24h_pairs_velocity/HV/region-27_0-82_951-35_52-83_8/2014/11/07/20141106T0905__20141107T0807_past.npz'
# future_file = '/path/to/SAR_sea_ice_dataset/VECTOR_FIELDS_24h_pairs_velocity/HV/region-27_0-82_951-35_52-83_8/2014/11/07/20141107T0807__20141108T0848_future.npz'

In [ ]:
# # future_file = '/path/to/SAR_sea_ice_dataset/VECTOR_FIELDS_24h/HV/region-33_07-82_34-40_1-83_297/2014/12/28/20141228T0654__20141229T0735_future.npz'
# past_file = '/path/to/SAR_sea_ice_dataset/VECTOR_FIELDS_24h/HV/region-33_0-82_95-40_715-83_903/2020/02/19/20200218T0653__20200219T0734_past.npz'
# future_file = '/path/to/SAR_sea_ice_dataset/VECTOR_FIELDS_24h/HV/region-33_0-82_95-40_715-83_903/2020/02/19/20200219T0734__20200220T0637_future.npz'

In [ ]:
# only 7 feature matches, but drift still seems reasonable
# future_file = '/path/to/SAR_sea_ice_dataset/VECTOR_FIELDS_24h_pairs/HV/region-33_0-82_95-40_715-83_903/2015/04/12/20150412T0807__20150413T0711_future.npz'

In [ ]:
# # few features
# past_file = '/path/to/SAR_sea_ice_dataset/VECTOR_FIELDS_24h_pairs/HV/region-33_07-82_34-40_1-83_297/2015/07/24/20150723T0719__20150724T0621_past.npz'
# # 7,1681,3
# future_file = '/path/to/SAR_sea_ice_dataset/VECTOR_FIELDS_24h_pairs/HV/region-33_07-82_34-40_1-83_297/2015/07/24/20150724T0621__20150725T0702_future.npz'
# # 8,1681,5

In [ ]:
# # many features
# past_file = '/path/to/SAR_sea_ice_dataset/VECTOR_FIELDS_24h_pairs/HV/region-33_07-82_34-40_1-83_297/2014/12/27/20141226T1342__20141227T1245_past.npz'
# # 4652,1681,1134
# future_file = '/path/to/SAR_sea_ice_dataset/VECTOR_FIELDS_24h_pairs/HV/region-33_07-82_34-40_1-83_297/2014/12/27/20141227T1245__20141228T1326_future.npz'
# # 4464,1681,1189

In [ ]:
# past_file = '/path/to/SAR_sea_ice_dataset/VECTOR_FIELDS_24h_pairs/HV/region-38_65-82_33-45_61-83_293/2018/04/20/20180419T0637__20180420T0717_past.npz'
# future_file = '/path/to/SAR_sea_ice_dataset/VECTOR_FIELDS_24h_pairs/HV/region-38_65-82_33-45_61-83_293/2018/04/20/20180420T0717__20180421T0620_future.npz'

In [ ]:
import json

past_paths = []
future_paths = []

jsonl_path = "/path/to/project/model_dev_main/index_files/min400PM_wind_drift_SAR_dataset/index_val.jsonl"

with open(jsonl_path, "r") as f:
    for line in f:
        row = json.loads(line)
        past_paths.append(row["past_drift_path"])
        future_paths.append(row["future_drift_path"])

print(f"Found {len(past_paths)} past drift paths")
print(f"Found {len(future_paths)} future drift paths")

In [ ]:
# for jsonl_path = "/path/to/project/model_dev_main/index_files/min400PM_wind_drift_SAR_dataset/index_test.jsonl"
# i=8 # divergence case of two leads opening
# i=11 # narrowing og a lead in the bottom right
# i=42 # future field has nice complex cirular motion


# i=9 # leands openinig
# i = 510 # pretty hidden leads opening
# i = 966 # good case for divergence loss prediction
i = 635
past_file = past_paths[i]
past_file = past_file.replace("_velocity", "")

future_file = future_paths[i]
future_file = future_file.replace("_velocity", "")

print(past_file)

In [ ]:
plot_drift(past_file, quiver_step=40)

In [ ]:
plot_drift(future_file, quiver_step=40)
plot_drift_divergence(future_file)
plot_drift_divergence_with_mask(future_file, pixel_size_m=100.0, mask_q=0.90)

In [ ]:
from pathlib import Path
import re

hv_dir = Path("/path/to/SAR_sea_ice_dataset/VECTOR_FIELDS_24h_pairs_velocity/HV")

past_count = 0
future_count = 0
pair_count = 0

# Matches:
#   <start>__<end>_past.npz
#   <start>__<end>_future.npz
# where <start>/<end> look like 20201028T0734 etc.
pat = re.compile(r"^(?P<start>\d{8}T\d{4})__(?P<end>\d{8}T\d{4})_(?P<tag>past|future)$")

# Group files by their parent folder
by_folder = {}
for p in hv_dir.rglob("*.npz"):
    m = pat.match(p.stem)  # stem = filename without .npz
    if not m:
        continue

    tag = m.group("tag")
    if tag == "past":
        past_count += 1
    else:
        future_count += 1

    folder = p.parent
    by_folder.setdefault(folder, {"past": [], "future": []})[tag].append(
        (m.group("start"), m.group("end"), p.name)
    )

# Count valid pairs in the same folder:
# last date/time in past filename == first date/time in future filename
for folder, d in by_folder.items():
    past_ends = {}
    for start, end, fname in d["past"]:
        past_ends.setdefault(end, []).append(fname)

    for start, end, fname in d["future"]:
        if start in past_ends:
            # Count one pair per matching (past_end, future_start) combination.
            # If there are multiple pasts with same end, count them all.
            pair_count += len(past_ends[start])

print(f"past files:   {past_count}")
print(f"future files: {future_count}")
print(f"valid pairs (past end == future start, same folder): {pair_count}")

